# MethaneScope AI — Lesson 2.4
## PostgreSQL + PostGIS: a spatial database for satellite methane observations

**Learning outcomes**
1. Explain the difference between raster files, geographic *footprints*, and plume-outline geometries.
2. Start an isolated local PostgreSQL/PostGIS service with Docker Compose.
3. Design relational tables, primary/foreign keys, uniqueness constraints, `geometry` columns and spatial indexes.
4. Import only the `ACCEPTED` records from Lesson 2.3 after independently rechecking their input files.
5. Demonstrate **idempotency**: running the ingestion twice does not create duplicate granules.
6. Execute your first SQL/PostGIS geographical queries.

**Scientific safety:** EMITL2BCH4PLM.002 raster data are methane column **enhancements (ppm m)**. They are not automatically emission rates (kg/h). "ACCEPTED" means Lesson 2.3 structural checks passed; it does not establish scientific validity. "Demo" is a synthetic test fixture, not NASA data. This notebook preserves that distinction.

**Prerequisite:** Run Lesson 2.3, either in `MODE = "demo"` or in `MODE = "real"`. It should have created `outputs/emit_batch/emit_accepted_for_ingestion.csv`. The source GeoTIFF/GeoJSON files must still be present.

**Files included with this lesson:** `docker-compose.postgis.yml` and `config/postgis.env.example`. Follow the setup notes before executing the database connection cell.

### 0 — One-time setup (Git Bash on Windows)

1. Install and start **Docker Desktop**, with Linux containers enabled. If it is not installed, stop here and ask for a native PostgreSQL/PostGIS alternative; the Python environment alone does not provide the database server.
2. In the MethaneScope project directory, place `docker-compose.postgis.yml` in the root and `config/postgis.env.example` in `config/`.
3. Create local environment settings (this includes a **local development-only password**):

```bash
cp config/postgis.env.example .env
```

4. Launch the database, then verify:

```bash
docker compose -f docker-compose.postgis.yml up -d
docker compose -f docker-compose.postgis.yml ps
```

5. Install the Python database driver **inside your existing virtual environment**, by running the next cell. Note: `.env` contains credentials—do not commit it to Git or share screenshots of it. The Compose port binds to **127.0.0.1 only**, to avoid exposing your development database on the network.

To stop: `docker compose -f docker-compose.postgis.yml down`. This retains your data volume. **Do not run `down -v`** unless you intend to delete the database volume.

### 1 — Install client libraries

Run the following cell once. If you prefer your Git Bash terminal, use:

```bash
python -m pip install "psycopg[binary]>=3.2,<4" python-dotenv pandas rasterio shapely
```

In [ ]:
%pip install "psycopg[binary]>=3.2,<4" python-dotenv pandas rasterio shapely

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import re

import pandas as pd
import psycopg
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import shape
from dotenv import load_dotenv
import os

# Locate the MethaneScope root even when notebook is inside notebooks/
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "outputs" / "emit_batch").exists()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Lesson 2.3 output folder not found. Run Lesson 2.3 first, "
        "or open JupyterLab from your MethaneScope project directory."
    )

if not load_dotenv(ROOT / ".env"):
    raise FileNotFoundError(
        f"Missing {ROOT / '.env'}. Run: cp config/postgis.env.example .env"
    )

MANIFEST = ROOT / "outputs" / "emit_batch" / "emit_accepted_for_ingestion.csv"
print("Project root:", ROOT)
print("Accepted manifest present:", MANIFEST.is_file())

### 2 — Connect to our local PostgreSQL database

Never print the password. Docker Compose creates a PostgreSQL database and installs the PostGIS extension through the notebook in the next step. This lesson's role is for **local development**, not a production-security template.

In [ ]:
connection_settings = {
    "host": os.getenv("DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("DB_PORT", "5433")),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
    "connect_timeout": 8,
    "autocommit": True,
}

try:
    conn = psycopg.connect(**connection_settings)
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user, version()")
        db_info = cur.fetchone()
    print("Connected database:", db_info[0])
    print("Connected role:", db_info[1])
    print("PostgreSQL server:", db_info[2].split(",")[0])
except psycopg.OperationalError as exc:
    raise RuntimeError(
        "PostgreSQL connection failed. Is Docker Desktop running, and "
        "did 'docker compose -f docker-compose.postgis.yml up -d' succeed? "
        "Verify .env settings and port 5433."
    ) from exc

### 3 — Create a spatial database schema

**Tables**
- `data_sources`: provenance and whether the collection is synthetic or real.
- `satellite_granules`: logical granule identity, checksums, summary statistics and a **geographical raster footprint** (not a detected plume outline).
- `plume_features`: actual geometries read from each file's GeoJSON `features`, with their original properties.
- `ingestion_runs`: audits counts and failures for each execution.

**Constraints:** Two separate unique constraints protect against the *same ID with different bytes* and *different IDs with identical raster bytes*. Foreign keys preserve the link between plumes and granules. GiST indexes support spatial search.

In [ ]:
SCHEMA_SQL = """
CREATE EXTENSION IF NOT EXISTS postgis;

CREATE TABLE IF NOT EXISTS data_sources (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_code TEXT NOT NULL UNIQUE,
    product TEXT NOT NULL,
    description TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS satellite_granules (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_id BIGINT NOT NULL REFERENCES data_sources(id),
    granule_id TEXT NOT NULL,
    acquisition_time TIMESTAMPTZ,
    raster_sha256 CHAR(64) NOT NULL,
    geojson_sha256 CHAR(64),
    raster_path TEXT NOT NULL,
    geojson_path TEXT NOT NULL,
    valid_pixels INTEGER NOT NULL CHECK (valid_pixels > 0),
    max_enhancement_ppm_m DOUBLE PRECISION NOT NULL,
    data_origin TEXT NOT NULL,
    footprint geometry(Polygon, 4326) NOT NULL,
    raw_manifest JSONB NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    CONSTRAINT uq_source_granule UNIQUE (source_id, granule_id),
    CONSTRAINT uq_source_raster_sha UNIQUE (source_id, raster_sha256),
    CONSTRAINT ck_footprint_valid CHECK (ST_IsValid(footprint))
);

CREATE INDEX IF NOT EXISTS idx_granule_footprint
ON satellite_granules USING GIST (footprint);
CREATE INDEX IF NOT EXISTS idx_granule_acquisition
ON satellite_granules (acquisition_time);

CREATE TABLE IF NOT EXISTS plume_features (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    granule_pk BIGINT NOT NULL REFERENCES satellite_granules(id) ON DELETE CASCADE,
    feature_index INTEGER NOT NULL CHECK (feature_index >= 0),
    geom geometry(Geometry, 4326) NOT NULL,
    properties JSONB NOT NULL,
    CONSTRAINT uq_granule_feature UNIQUE (granule_pk, feature_index),
    CONSTRAINT ck_feature_valid CHECK (ST_IsValid(geom))
);
CREATE INDEX IF NOT EXISTS idx_plume_geom
ON plume_features USING GIST (geom);

CREATE TABLE IF NOT EXISTS ingestion_runs (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    started_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    source_mode TEXT NOT NULL CHECK(source_mode IN ('demo', 'real')),
    attempted INTEGER NOT NULL,
    inserted INTEGER NOT NULL,
    skipped INTEGER NOT NULL,
    rejected INTEGER NOT NULL,
    details JSONB NOT NULL
);
"""

with conn.transaction():
    with conn.cursor() as cur:
        # DDL is trusted, fixed notebook source code (no user string interpolation)
        for statement in SCHEMA_SQL.split(";"):
            if statement.strip():
                cur.execute(statement)
        cur.execute("SELECT PostGIS_Version()")
        print("PostGIS version:", cur.fetchone()[0])

print("Tables and indexes created (or already present).")

### 4 — Read the Lesson 2.3 manifest

We deliberately **do not reclassify** `REVIEW`, `REJECTED`, `CONFLICT` or `DUPLICATE` as accepted. We also check the current file bytes again, so changes since Lesson 2.3 do not go unnoticed. For a proper production system the validation schema would be versioned and stored with each result.

In [ ]:
if not MANIFEST.is_file():
    raise FileNotFoundError(
        f"Missing {MANIFEST}. Re-run the export cells in Lesson 2.3 first."
    )

accepted = pd.read_csv(MANIFEST, keep_default_na=False)
required_columns = {
    "mode", "status", "data_origin", "granule_id", "path",
    "raster_sha256", "geojson_path", "geojson_sha256",
    "valid_pixels", "maximum_enhancement_ppm_m"
}
missing = required_columns - set(accepted.columns)
if missing:
    raise ValueError(f"Unexpected Lesson 2.3 CSV format, missing: {sorted(missing)}")
if accepted.empty:
    raise ValueError(
        "No ACCEPTED records. Inspect outputs/emit_batch/emit_batch_issues.json "
        "and fix the validation issues first."
    )
if not accepted["status"].eq("ACCEPTED").all():
    raise ValueError("Import blocked: manifest contains non-ACCEPTED records")
if accepted["mode"].nunique() != 1 or accepted["mode"].iloc[0] not in ("demo", "real"):
    raise ValueError("Import blocked: mixed or unknown demonstration/real mode")
MODE = str(accepted["mode"].iloc[0])
SOURCE_CODE = "SYNTHETIC_EMIT_QA" if MODE == "demo" else "NASA_EMIT_CANDIDATE"
print("Import mode:", MODE)
print("Source identity:", SOURCE_CODE)
print("Eligible rows:", len(accepted))
display(accepted[["granule_id", "status", "valid_pixels", "maximum_enhancement_ppm_m"]].head(10))

### 5 — Define safe provenance and geometry helpers

The raster footprint is computed by reprojecting its bounds to `EPSG:4326`. This is **not** the methane-plume outline or the exact location of the emitting facility. Separately, GeoJSON `Feature` geometries will be saved without inventing what each feature represents.

In [ ]:
def safe_project_file(relative_path):
    if not relative_path or not isinstance(relative_path, str):
        raise ValueError("A source file path is missing")
    candidate = Path(relative_path)
    if candidate.is_absolute() or ".." in candidate.parts:
        raise ValueError(f"Unsafe path in manifest: {relative_path!r}")
    resolved = (ROOT / candidate).resolve()
    if not resolved.is_relative_to(ROOT.resolve()):
        raise ValueError("Path escapes the MethaneScope project directory")
    if not resolved.is_file():
        raise FileNotFoundError(f"Source file missing: {relative_path}")
    return resolved


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def parse_utc_from_granule(granule_id):
    match = re.search(r"_(\d{8}T\d{6})_", granule_id)
    if match is None:
        raise ValueError(f"Cannot determine acquisition time from {granule_id}")
    return datetime.strptime(match.group(1), "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)


def get_raster_bounds_wgs84(path):
    with rasterio.open(path) as src:
        if src.count != 1 or src.crs is None:
            raise ValueError("Expected a single-band raster with known CRS")
        left, bottom, right, top = transform_bounds(
            src.crs, "EPSG:4326", *src.bounds, densify_pts=21
        )
    vals = [left, bottom, right, top]
    if not all(math.isfinite(v) for v in vals):
        raise ValueError("Non-finite geographic bounds")
    if not (-180 <= left < right <= 180 and -90 <= bottom < top <= 90):
        raise ValueError("Unexpected geographic bounds or antimeridian crossing")
    return vals


def read_geojson_features(path):
    doc = json.loads(path.read_text(encoding="utf-8"))
    if doc.get("type") != "FeatureCollection":
        raise ValueError("Expected GeoJSON FeatureCollection")
    feats = doc.get("features")
    if not isinstance(feats, list) or not feats or len(feats) > 10000:
        raise ValueError("GeoJSON must have 1–10000 features")
    clean = []
    for i, feat in enumerate(feats):
        if feat.get("type") != "Feature" or not feat.get("geometry"):
            raise ValueError(f"Feature {i} is missing a geometry")
        geom = feat["geometry"]
        geometry = shape(geom)
        if geometry.is_empty or not geometry.is_valid:
            raise ValueError(f"Feature {i} geometry is invalid")
        west, south, east, north = geometry.bounds
        if not (-180 <= west <= east <= 180 and -90 <= south <= north <= 90):
            raise ValueError(f"Feature {i} coordinates outside WGS84 limits")
        clean.append((i, json.dumps(geom), json.dumps(feat.get("properties") or {})))
    return clean

print("Validation helpers ready")

### 6 — Insert source identities and ingest files transactionally

A single source granule and all its GeoJSON features are inserted in **one transaction**, so partial records will not remain after an error. Unchanged repeat ingestions are skipped. Conflicting IDs or hashes are reported, never silently overwritten. The checksum verifies that files did not change after the manifest was written.

**Safety boundary:** This is educational ingestion of structurally accepted candidate data. It is not an official NASA ingest, nor proof that a measurement or emission estimate is scientifically validated.

In [ ]:
with conn.transaction():
    with conn.cursor() as cur:
        cur.execute(
            """INSERT INTO data_sources (source_code, product, description)
               VALUES (%s,%s,%s) ON CONFLICT (source_code) DO NOTHING""",
            (
                SOURCE_CODE, "EMITL2BCH4PLM.002",
                "Synthetic Lesson 2.3 QA fixtures" if MODE == "demo"
                else "NASA EMIT candidate files; catalogue identity not independently verified",
            )
        )
        cur.execute("SELECT id FROM data_sources WHERE source_code=%s", (SOURCE_CODE,))
        SOURCE_ID = cur.fetchone()[0]

print("Source registry ID:", SOURCE_ID)


def import_one(row):
    r = row.to_dict()
    tif = safe_project_file(str(r["path"]))
    geo = safe_project_file(str(r["geojson_path"]))
    raster_sha = sha256_file(tif)
    geo_sha = sha256_file(geo)

    if raster_sha != str(r["raster_sha256"]) or geo_sha != str(r["geojson_sha256"]):
        raise ValueError("Source file changed since Lesson 2.3: checksum mismatch")

    granule_id = str(r["granule_id"])
    acquisition_time = parse_utc_from_granule(granule_id)
    bounds = get_raster_bounds_wgs84(tif)
    features = read_geojson_features(geo)
    valid_pixels = int(r["valid_pixels"])
    max_ppm_m = float(r["maximum_enhancement_ppm_m"])
    if valid_pixels <= 0 or not math.isfinite(max_ppm_m):
        raise ValueError("Invalid raster summary in accepted manifest")

    with conn.transaction():
        with conn.cursor() as cur:
            cur.execute(
                """SELECT id, granule_id, raster_sha256, geojson_sha256
                   FROM satellite_granules
                   WHERE source_id=%s AND (granule_id=%s OR raster_sha256=%s)""",
                (SOURCE_ID, granule_id, raster_sha)
            )
            existing = cur.fetchall()
            if existing:
                if len(existing) == 1:
                    ex = existing[0]
                    if (ex[1] == granule_id and ex[2].strip() == raster_sha
                            and (ex[3] or "").strip() == geo_sha):
                        return "SKIPPED", granule_id, "Identical granule already ingested"
                return "CONFLICT", granule_id, "Conflicting granule ID or raster checksum"

            cur.execute(
                """INSERT INTO satellite_granules
                 (source_id, granule_id, acquisition_time, raster_sha256,
                  geojson_sha256, raster_path, geojson_path, valid_pixels,
                  max_enhancement_ppm_m, data_origin, footprint, raw_manifest)
                 VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
                         ST_MakeEnvelope(%s,%s,%s,%s,4326),%s::jsonb)
                 ON CONFLICT DO NOTHING RETURNING id""",
                (
                    SOURCE_ID, granule_id, acquisition_time, raster_sha, geo_sha,
                    str(r["path"]), str(r["geojson_path"]), valid_pixels,
                    max_ppm_m, str(r["data_origin"]), *bounds,
                    json.dumps({key:(None if pd.isna(val) else val)
                                for key, val in r.items()}, default=str)
                )
            )
            result = cur.fetchone()
            if result is None:
                return "CONFLICT", granule_id, "Unique key blocked insert (concurrent duplicate or conflict)"
            granule_pk = result[0]
            for index, geometry_json, properties_json in features:
                cur.execute(
                    """INSERT INTO plume_features
                       (granule_pk, feature_index, geom, properties)
                       VALUES (%s,%s,ST_SetSRID(ST_GeomFromGeoJSON(%s),4326),%s::jsonb)""",
                    (granule_pk, index, geometry_json, properties_json)
                )
    return "INSERTED", granule_id, f"{len(features)} GeoJSON feature(s) stored"

results = []
for _, row in accepted.iterrows():
    try:
        status, granule, message = import_one(row)
    except Exception as exc:
        status, granule, message = "ERROR", str(row.get("granule_id")), str(exc)
    results.append({"status":status, "granule_id":granule, "message":message})
    print(f"{status:10} {granule}: {message[:120]}")

result_table = pd.DataFrame(results)
display(result_table)

### 7 — Write an audit record and query the spatial database

These queries use real PostGIS geometry operations: `ST_AsGeoJSON` to serialize a footprint; `ST_Centroid` to estimate its geometric centre; and `ST_Intersects` for spatial filtering. **Centroid of raster footprint ≠ methane source location.**

In [ ]:
counts = result_table["status"].value_counts().to_dict()
with conn.transaction():
    with conn.cursor() as cur:
        cur.execute(
            """INSERT INTO ingestion_runs
               (source_mode, attempted, inserted, skipped, rejected, details)
               VALUES (%s,%s,%s,%s,%s,%s::jsonb) RETURNING id""",
            (
                MODE, len(results), counts.get("INSERTED", 0), counts.get("SKIPPED", 0),
                counts.get("ERROR", 0) + counts.get("CONFLICT", 0),
                json.dumps(results)
            )
        )
        print("Ingestion audit ID:", cur.fetchone()[0])

with conn.cursor() as cur:
    cur.execute("""
        SELECT g.granule_id, g.acquisition_time,
               g.max_enhancement_ppm_m,
               ST_X(ST_Centroid(g.footprint)) AS footprint_centre_lon,
               ST_Y(ST_Centroid(g.footprint)) AS footprint_centre_lat,
               (SELECT count(*) FROM plume_features f WHERE f.granule_pk=g.id) AS geojson_features
        FROM satellite_granules g
        WHERE g.source_id=%s
        ORDER BY g.acquisition_time, g.granule_id
        LIMIT 30
    """, (SOURCE_ID,))
    columns = [c.name for c in cur.description]
    rows = cur.fetchall()
print("Stored granules for this source:")
display(pd.DataFrame(rows, columns=columns))

### 8 — Spatial query: find granules overlapping an area

The box in this example is an **arbitrary demonstration bounding box** in lon/lat degrees; it is not a verified emitting site. For real NASA files, adapt these coordinates to the area of interest.

In [ ]:
# Bounding box around a demonstration region (WGS84 longitude and latitude)
west, south, east, north = 27.0, -19.0, 32.0, -16.0
with conn.cursor() as cur:
    cur.execute("""
        SELECT granule_id,
               ROUND(ST_Area(footprint::geography)/1000000.0) AS footprint_area_km2
        FROM satellite_granules
        WHERE source_id=%s AND ST_Intersects(
            footprint, ST_MakeEnvelope(%s,%s,%s,%s,4326)
        )
        ORDER BY granule_id
    """, (SOURCE_ID, west, south, east, north))
    found = cur.fetchall()

print("Granules overlapping example search area:", len(found))
for granule, area in found:
    print(granule, "footprint area (km²):", area)

### 9 — Verify idempotency (repeat the same ingestion)

An idempotent importer should **not increase** the number of granules or plume features when given an unchanged manifest twice. We rerun the same import helper and assert that counts remain constant. A change in file content should be flagged as a checksum mismatch, not silently ingested.

In [ ]:
def database_counts():
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM satellite_granules WHERE source_id=%s", (SOURCE_ID,))
        granules_count = cur.fetchone()[0]
        cur.execute("""
            SELECT count(*) FROM plume_features f
            JOIN satellite_granules g ON g.id=f.granule_pk
            WHERE g.source_id=%s
        """, (SOURCE_ID,))
        features_count = cur.fetchone()[0]
    return granules_count, features_count

before = database_counts()
second_pass = []
for _, row in accepted.iterrows():
    try:
        second_pass.append(import_one(row)[0])
    except Exception as exc:
        second_pass.append("ERROR: " + str(exc))
after = database_counts()

print("Before second pass:", before)
print("After second pass: ", after)
print("Second-pass outcomes:", second_pass)
assert before == after, "Idempotency failed: unexpected additional database rows"
assert all(s == "SKIPPED" for s in second_pass), "Review outcomes: at least one file did not skip"
print("PASS: repeated ingestion did not duplicate data")

### 10 — Save a local audit summary and close the connection

In [ ]:
output_dir = ROOT / "outputs" / "postgis"
output_dir.mkdir(parents=True, exist_ok=True)
result_csv = output_dir / "lesson_2_4_ingestion_outcomes.csv"
result_table.to_csv(result_csv, index=False)

summary = {
    "lesson": "2.4",
    "mode": MODE,
    "source": SOURCE_CODE,
    "ingestion_counts": counts,
    "granules_after_idempotency_test": database_counts()[0],
    "plume_features_after_idempotency_test": database_counts()[1],
    "scientific_validation": "NOT_ESTABLISHED",
    "export_file": str(result_csv.relative_to(ROOT)),
}
summary_path = output_dir / "lesson_2_4_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
conn.close()
print("Lesson 2.4 complete. Audits saved:")
print(result_csv)
print(summary_path)

---
## Knowledge checkpoint

1. Why is a **raster footprint** different from a plume outline or emitting source location?
2. What is the difference between a **primary key**, **foreign key**, and **unique constraint**?
3. What does the GiST index help the database do?
4. How would our system react if a file kept the same `granule_id` but its contents changed?
5. Why do we distinguish **synthetic test data** from **NASA candidate observations**?
6. Why does a `ppm m` raster value not by itself represent `kg/h` of emissions?

**What to send me:** the PostgreSQL/PostGIS version line, the ingestion outcomes table, the PostGIS spatial query result, and `PASS: repeated ingestion did not duplicate data`.

### Next lesson — 2.5
Turn this notebook into a reusable Python ingestion module and add automated tests, quality issue logs, scheduling and a repeatable command-line pipeline. After that, we will begin integrating high-frequency plume events and source identification.